# Week 49, Day 2: Review: trees and heaps

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w049d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class T:
    def __init__(self, v, l=None, r=None):
        self.v, self.l, self.r = v, l, r

root = T(1, T(2, T(4), T(5)), T(3, None, T(6)))

def pre(n):
    return [] if n is None else [n.v] + pre(n.l) + pre(n.r)

def ino(n):
    return [] if n is None else ino(n.l) + [n.v] + ino(n.r)

def post(n):
    return [] if n is None else post(n.l) + post(n.r) + [n.v]

print(pre(root), ino(root), post(root))

In [ ]:
from collections import deque

class T:
    def __init__(self, v, l=None, r=None):
        self.v, self.l, self.r = v, l, r

def levels(root):
    out, queue = [], deque([root] if root else [])
    while queue:
        out.append([n.v for n in queue])
        queue = deque(c for n in queue for c in (n.l, n.r) if c)
    return out

def inorder(root):
    out, stack, node = [], [], root
    while stack or node:
        while node:
            stack.append(node)
            node = node.l
        node = stack.pop()
        out.append(node.v)
        node = node.r
    return out

root = T(1, T(2, T(4), T(5)), T(3, None, T(6)))
print(levels(root), inorder(root))

In [ ]:
class T:
    def __init__(self, v, l=None, r=None):
        self.v, self.l, self.r = v, l, r

def max_path_sum(root):
    best = [float("-inf")]                  # a one-item list as a holder (Phase 5 has nonlocal)
    def down_from(n):
        if n is None:
            return 0
        left = max(0, down_from(n.l))
        right = max(0, down_from(n.r))
        best[0] = max(best[0], n.v + left + right)      # through this node
        return n.v + max(left, right)                   # best path going down from it
    down_from(root)
    return best[0]

print(max_path_sum(T(-10, T(9), T(20, T(15), T(7)))), max_path_sum(T(-3)))

In [ ]:
class T:
    def __init__(self, v, l=None, r=None):
        self.v, self.l, self.r = v, l, r

def range_sum(node, lo, hi):
    total, stack = 0, [node]
    while stack:
        n = stack.pop()
        if n is None:
            continue
        if lo <= n.v <= hi:
            total += n.v
        if n.v > lo:
            stack.append(n.l)               # smaller values might be in range
        if n.v < hi:
            stack.append(n.r)
    return total

root = T(10, T(5, T(3), T(7)), T(15, None, T(18)))
print(range_sum(root, 7, 15))

In [ ]:
import heapq

def min_rooms(meetings):
    """The fewest rooms so no two overlapping meetings share one."""
    ends = []                                   # a min-heap of end times of rooms in use
    for start, end in sorted(meetings):
        if ends and ends[0] <= start:
            heapq.heapreplace(ends, end)        # reuse the room that frees up first
        else:
            heapq.heappush(ends, end)
    return len(ends)

print(min_rooms([(0, 30), (5, 10), (15, 20)]), min_rooms([(7, 10), (2, 4)]), min_rooms([(1, 5), (5, 9)]))

In [ ]:
def build(words):
    root = {}
    for w in words:
        node = root
        for ch in w:
            node = node.setdefault(ch, {})
        node["$"] = True
    return root

def has_prefix_word(root, s):
    """Is any stored word a prefix of s?"""
    node = root
    for ch in s:
        if "$" in node:
            return True
        node = node.get(ch)
        if node is None:
            return False
    return "$" in node

root = build(["cat", "do", "dove"])
print(has_prefix_word(root, "catalogue"), has_prefix_word(root, "dog"), has_prefix_word(root, "ca"))

> **Key idea.** Pick a traversal by what you need: pre-order for top-down work, in-order for sorted BST order, post-order when a node needs its children's answers, level order for anything by depth; use a stack or queue when trees may be tall. Classify tree problems by whether information flows down, up or both. BSTs give O(h) operations, sorted in-order, pruned range queries and bound-based validation; balance keeps h near log n. Heaps give top-k in O(n log k), priority queues with tie-breaking counters, k-way merges and running medians. For strings: tries for prefixes, KMP for one pattern, hashes for comparing substrings, Aho-Corasick for many patterns.

### Exercise: Levels and depths

Write `level_averages(root)` (the average value on each level, top first, with a queue) and `min_depth(root)`: the number of nodes on the shortest path from the root to a **leaf** (0 for an empty tree), stopping at the first leaf found by a breadth-first search. The checks use a 20,000-node chain.

In [ ]:
from collections import deque

class T:
    def __init__(self, v, l=None, r=None):
        self.v, self.l, self.r = v, l, r

def level_averages(root):
    ___

def min_depth(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_levels():\n    \"levels\"\n    root = T(3, T(9), T(20, T(15), T(7)))\n    assert level_averages(root) == [3.0, 14.5, 11.0] and level_averages(None) == []\n    assert min_depth(root) == 2 and min_depth(None) == 0 and min_depth(T(1, T(2))) == 2\n    chain = None\n    for v in range(20_000):\n        chain = T(v, chain)\n    assert min_depth(chain) == 20_000 and len(level_averages(chain)) == 20_000")

### Exercise: Down and up

Write `max_path_sum(root)` (the largest sum along any path between two nodes, at least one node; values may be negative) and `good_nodes(root)`: how many nodes have no larger value on the path from the root to them (information flowing down). Recursion is fine here: the checks' trees are at most 30 deep.

In [ ]:
class T:
    def __init__(self, v, l=None, r=None):
        self.v, self.l, self.r = v, l, r

def max_path_sum(root):
    ___

def good_nodes(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef rand_tree(rng, n):\n    if n == 0:\n        return None\n    k = rng.randint(0, n - 1)\n    return T(rng.randint(-10, 10), rand_tree(rng, k), rand_tree(rng, n - 1 - k))\ndef all_nodes(n):\n    return [] if n is None else [n] + all_nodes(n.l) + all_nodes(n.r)\ndef slow_max(root):\n    nodes = all_nodes(root)\n    parent = {}\n    for n in nodes:\n        for c in (n.l, n.r):\n            if c:\n                parent[id(c)] = n\n    def path_up(n):\n        out = [n]\n        while id(out[-1]) in parent:\n            out.append(parent[id(out[-1])])\n        return out\n    best = None\n    for a in nodes:\n        for b in nodes:\n            pa, pb = path_up(a), path_up(b)\n            common = next(x for x in pa if any(x is y for y in pb))\n            total = 0\n            for x in pa:\n                total += x.v\n                if x is common:\n                    break\n            for y in pb:\n                if y is common:\n                    break\n                total += y.v\n            best = total if best is None else max(best, total)\n    return best\ndef test_paths():\n    \"paths\"\n    assert max_path_sum(T(-10, T(9), T(20, T(15), T(7)))) == 42 and max_path_sum(T(-3)) == -3\n    assert good_nodes(T(3, T(1, T(3)), T(4, T(1), T(5)))) == 4 and good_nodes(None) == 0\n    rng = random.Random(1)\n    for _ in range(80):\n        t = rand_tree(rng, rng.randint(1, 9))\n        assert max_path_sum(t) == slow_max(t)")

### Exercise: Using the order

For a BST, write `range_sum(root, lo, hi)` (pruning subtrees that can't contribute, with a stack) and `kth_largest(root, k)` with a **reverse** in-order walk (right, node, left) that stops after k values (`IndexError` if there are fewer). The checks count node visits.

In [ ]:
class T:
    def __init__(self, v, l=None, r=None):
        self.v, self.l, self.r = v, l, r

def range_sum(root, lo, hi):
    ___

def kth_largest(root, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nclass C(T):\n    reads = 0\n    def __init__(self, v, l=None, r=None):\n        super().__init__(v, l, r)\n    def __getattribute__(self, name):\n        if name == \"v\":\n            C.reads += 1\n        return object.__getattribute__(self, name)\ndef balanced(lo, hi):\n    if lo > hi:\n        return None\n    m = (lo + hi) // 2\n    return C(m, balanced(lo, m - 1), balanced(m + 1, hi))\ndef test_bst():\n    \"order\"\n    root = T(10, T(5, T(3), T(7)), T(15, None, T(18)))\n    assert range_sum(root, 7, 15) == 32 and kth_largest(root, 1) == 18 and kth_largest(root, 4) == 7\n    try:\n        kth_largest(root, 7)\n        assert False\n    except IndexError:\n        pass\n    big = balanced(1, 4095)\n    C.reads = 0\n    assert range_sum(big, 10, 20) == sum(range(10, 21)) and C.reads < 200, C.reads\n    C.reads = 0\n    assert kth_largest(big, 3) == 4093 and C.reads < 100, C.reads")

### Exercise: Rooms and a heap

Write `min_rooms(meetings)` (as in "Heaps") and `assign_rooms(meetings)`: give each meeting `(start, end)` a room number, using the **lowest-numbered free room** when it starts (rooms are numbered from 0, a room is free again at its meeting's end time), and return the list of room numbers in the meetings' original order. Keep two heaps: rooms in use as `(end, room)`, and free room numbers.

In [ ]:
import heapq

def min_rooms(meetings):
    ___

def assign_rooms(meetings):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_rooms():\n    \"rooms\"\n    assert min_rooms([(0, 30), (5, 10), (15, 20)]) == 2 and min_rooms([]) == 0 and min_rooms([(1, 5), (5, 9)]) == 1\n    assert assign_rooms([(0, 30), (5, 10), (15, 20), (10, 12)]) == [0, 1, 1, 1]\n    assert assign_rooms([(5, 8), (0, 10), (1, 3)]) == [1, 0, 1]\n    rng = random.Random(2)\n    for _ in range(200):\n        ms = [(s, s + rng.randint(1, 8)) for s in (rng.randint(0, 20) for _ in range(rng.randint(0, 10)))]\n        rooms = assign_rooms(ms)\n        assert max(rooms, default=-1) + 1 == min_rooms(ms)\n        for i in range(len(ms)):\n            for j in range(len(ms)):\n                if i != j and rooms[i] == rooms[j]:\n                    assert ms[i][1] <= ms[j][0] or ms[j][1] <= ms[i][0], \"a clash\"")

### Exercise: No two the same in a row

Write `spread_out(s)`: rearrange the characters of s so no two neighbours are equal, or return `""` if that's impossible. Greedily take the most frequent remaining character that isn't the one just placed, using a `heapq` max-heap of `(-count, char)` (so ties go to the alphabetically earlier character), holding back the character just used for one step. Then `possible(s)`: whether it can be done, from the counts alone (the commonest character may appear at most `(len(s) + 1) // 2` times).

In [ ]:
import heapq
from collections import Counter

def spread_out(s):
    ___

def possible(s):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_spread():\n    \"spreading\"\n    assert spread_out(\"aab\") == \"aba\" and spread_out(\"aaab\") == \"\" and spread_out(\"\") == \"\"\n    assert spread_out(\"aaabbc\") == \"ababac\"\n    rng = random.Random(3)\n    for _ in range(300):\n        s = \"\".join(rng.choice(\"abc\") for _ in range(rng.randint(0, 10)))\n        r = spread_out(s)\n        assert possible(s) == (r != \"\" or s == \"\")\n        if r:\n            assert sorted(r) == sorted(s) and all(a != b for a, b in zip(r, r[1:]))")

### Exercise: Built letter by letter

Write `longest_buildable(words)`: the longest word that can be built one letter at a time from other words in the list (every prefix of it, from length 1, is also in the list); the alphabetically first on a tie; `""` if none. Build a trie with end marks, then search it with a stack, only going down into nodes that are word ends.

In [ ]:
def longest_buildable(words):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_build():\n    \"buildable words\"\n    assert longest_buildable([\"w\", \"wo\", \"wor\", \"worl\", \"world\"]) == \"world\"\n    assert longest_buildable([\"a\", \"banana\", \"app\", \"appl\", \"ap\", \"apply\", \"apple\"]) == \"apple\"\n    assert longest_buildable([\"abc\"]) == \"\" and longest_buildable([]) == \"\"\n    rng = random.Random(4)\n    for _ in range(200):\n        words = list({\"\".join(rng.choice(\"ab\") for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(0, 12))})\n        ok = [w for w in words if all(w[:i] in words for i in range(1, len(w)))]\n        want = min(ok, key=lambda w: (-len(w), w)) if ok else \"\"\n        assert longest_buildable(words) == want, words")

### Bug hunt: Only the children

This validates a BST by comparing each node only with its children, so a value deep on the wrong side of an ancestor slips through: this prints `True`. Pass bounds down so it prints `False`.

In [ ]:
class T:
    def __init__(self, v, l=None, r=None):
        self.v, self.l, self.r = v, l, r

def is_bst(n):
    if n is None:
        return True
    if n.l and n.l.v >= n.v:
        return False
    if n.r and n.r.v <= n.v:
        return False
    return is_bst(n.l) and is_bst(n.r)

print(is_bst(T(10, T(5, None, T(12)), T(15))))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"bounds\"\n    expect_output(\"False\")\n    assert is_bst(T(10, T(5, None, T(7)), T(15)))")

### Bug hunt: Ties that crash

This priority queue pushes `(priority, task)` where tasks are dicts, so two tasks with equal priority make `heapq` compare the dicts: it crashes with a `TypeError`. Add a counter as a tie-breaker so it prints `['wash', 'dry', 'fold']`.

In [ ]:
import heapq

heap = []
for p, name in [(1, "wash"), (2, "dry"), (2, "fold")]:
    heapq.heappush(heap, (p, {"name": name}))
print([heapq.heappop(heap)[-1]["name"] for _ in range(3)])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a tie-breaker\"\n    expect_output(\"['wash', 'dry', 'fold']\")")

### Bug hunt: The wrong heap for the largest

To keep the 3 largest values, this keeps a heap of size 3 but replaces the top whenever a newcomer is **smaller**, so it ends with small values: this prints `[3, 2, 1]`. Replace the top only when the newcomer is larger, so it prints `[9, 8, 7]`.

In [ ]:
import heapq

best = []
for x in [5, 1, 9, 3, 7, 2, 8]:
    if len(best) < 3:
        heapq.heappush(best, x)
    elif x < best[0]:
        heapq.heapreplace(best, x)
print(sorted(best, reverse=True))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"larger gets in\"\n    expect_output(\"[9, 8, 7]\")")

### Puzzle: In-order with a stack

Arrange the lines into an iterative in-order traversal. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the post-order of the tree 1(2(4, 5), 3(-, 6)), as in the warm-up
q = ___   # the height (in edges) of a perfect binary tree with 15 nodes
r = ___   # the list [5, 3, 8, 1] after heapq.heapify
s = ___   # the prefix function of "aabaa"
t = ___   # the root of an AVL tree after inserting 1, 2, 3 in that order
u = ___   # the number of nodes, counting the root, in a trie of "a", "ab", "b"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [4, 5, 2, 6, 3, 1] and q == 3 and r == [1, 3, 8, 5], \"p, q and r\"\n    assert s == [0, 1, 0, 1, 2] and t == 2 and u == 4, \"s, t and u\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: tasks with a cool-down

A machine runs one task per tick. Tasks are letters; the same letter must be separated by at least `n` other ticks (tasks or idle). Write `schedule(tasks, n)`: an optimal order as a string, using `"_"` for an idle tick, built greedily: at each tick, run the **most frequent** remaining task that isn't cooling down (ties alphabetical), or idle if none is available and tasks remain. Keep a `heapq` max-heap of `(-count, letter)` for ready tasks and a `deque` of `(ready_at, -count, letter)` for cooling ones. Then `min_ticks(tasks, n)`, the length of the schedule, from the formula `max(len(tasks), (maxcount - 1) * (n + 1) + number_of_letters_with_maxcount)`, and check it matches your schedule.

In [ ]:
import heapq
from collections import Counter, deque

def schedule(tasks, n):
    ___

def min_ticks(tasks, n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef valid(order, tasks, n):\n    last = {}\n    for i, ch in enumerate(order):\n        if ch != \"_\":\n            if ch in last and i - last[ch] <= n:\n                return False\n            last[ch] = i\n    return sorted(order.replace(\"_\", \"\")) == sorted(tasks)\ndef test_schedule():\n    \"schedules\"\n    assert schedule(\"AAABBB\", 2) == \"AB_AB_AB\" and min_ticks(\"AAABBB\", 2) == 8\n    assert schedule(\"AAABBB\", 0) == \"ABABAB\" and schedule(\"\", 3) == \"\" and min_ticks(\"\", 3) == 0\n    assert len(schedule(\"AAAABBBEEFFGG\", 3)) == min_ticks(\"AAAABBBEEFFGG\", 3) == 13\n    rng = random.Random(5)\n    for _ in range(300):\n        tasks = \"\".join(rng.choice(\"ABCD\") for _ in range(rng.randint(0, 14)))\n        n = rng.randint(0, 4)\n        order = schedule(tasks, n)\n        assert valid(order, tasks, n) and len(order) == min_ticks(tasks, n), (tasks, n, order)")

## Warm-up

## Traversals

Week 45's map of traversals:

| order | visits | good for |
|---|---|---|
| pre-order | node, left, right | copying, saving, prefix expressions, paths from the root |
| in-order | left, node, right | sorted order in a BST |
| post-order | left, right, node | sizes, heights, deleting, anything needing the children's answers first |
| level order | level by level, with a queue | shortest distances, views, widths, anything "by depth" |

Recursion is shortest, but Python's recursion limit (about 1,000) breaks it on tall trees, so for trees of unknown shape use an **explicit stack** (depth-first) or a **queue** (breadth-first). Each is O(n) time; a stack needs O(h) space and a queue O(width).

## Down, up, or both

Week 45 Day 5 sorted tree problems by which way information flows:

- **Down** (pre-order, passing parameters): depth, the path so far, bounds for BST validation, a running sum.
- **Up** (post-order, returning values): size, height, balance, the best path through a subtree.
- **Both**: a value passed down and a result returned up, like "longest path through any node" (return the best downward path, update a global best with the sum through this node).

## Search trees

Week 46: a BST keeps smaller values left and larger right, so search, insert and delete are O(h): O(log n) if balanced, O(n) if not. Inserting sorted data makes a chain; **AVL rotations** (or building from the middle of sorted data) keep h near log n. The BST property gives:

- **In-order is sorted**: kth smallest, validation, converting to a sorted list.
- **Pruning**: range queries and range sums skip subtrees that can't contain answers.
- **Floor, ceiling, successor**: remember the best candidate on the search path.
- **Validation** passes bounds down; checking only children is wrong.

## Heaps

Week 47: a binary heap is a complete tree in a list with every parent no larger than its children; push and pop are O(log n), `heap[0]` is O(1), and `heapify` is O(n). `heapq` works on plain lists, as a min-heap. The patterns:

- **Top k** of n: a min-heap of size k, O(n log k).
- **Always the most urgent next**: priority queues, simulations, scheduling; use `(priority, counter, item)` so ties never compare items.
- **Merging k sorted sources**: a heap of each source's front item, O(n log k).
- **Two heaps facing each other**: the running median.

## Strings

Week 48's tools, and when to reach for each:

- **Trie**: many words, questions about **prefixes** (autocomplete, word games, "is any stored word a prefix of this?").
- **Prefix function / KMP**: one pattern in a text or stream in O(n + m); also periods and borders.
- **Prefix hashes**: compare any two substrings in O(1); with binary search, longest repeated or common substrings.
- **Centres or Manacher**: palindromic substrings.
- **Aho-Corasick**: many patterns in one pass.
- For everyday searching, Python's `in`, `find` and `re` are fast and correct.

## Practice

## Quiz

## Daily challenge

## Before you go

Write, from memory, one sentence for each structure from Weeks 41 to 48 saying what it's best at. Tomorrow: the first timed set.